<a href="https://colab.research.google.com/github/Sergiodzm99/Proyect-Model_MLB/blob/main/DF_SAVANT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='00FF00'> PARQUET SAVANT

### <font color='00FFFF'> Extrer SAVANT
Se descargo el archivo parquet del codigo [BASE SACAT](https://colab.research.google.com/drive/1DSnlJHeEuPxSQEqnA0KrgVl240QUmmJH?authuser=1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_savant = pd.read_parquet('/content/drive/MyDrive/1. Cursos/1. UNIR/Master IA/4to Semestre/Proyecto Final/Base de Datos/Datos SAVAT/df_savant_2023_2026.parquet')

In [ ]:
df_savant

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2023-04-02,95.4,-1.65,5.69,"Hensley, David",682073,605182,field_out,hit_into_play,...,1.0,0.94,0.70,0.70,41.2,NaN,NaN,NaN,NaN,NaN
1,SI,2023-04-02,92.4,1.21,5.44,"Stott, Bryson",681082,527048,single,hit_into_play,...,1.0,2.01,1.39,1.39,43.6,NaN,NaN,NaN,NaN,NaN
2,SI,2023-04-02,92.5,1.24,5.44,"Stott, Bryson",681082,527048,None,called_strike,...,1.0,1.76,1.18,1.18,50.6,NaN,NaN,NaN,NaN,NaN
3,FF,2023-04-02,94.6,-1.76,5.69,"Hensley, David",682073,605182,None,foul,...,1.0,0.96,0.42,0.42,36.9,NaN,NaN,NaN,NaN,NaN
4,FF,2023-04-02,91.9,2.89,5.41,"Fletcher, David",664058,686610,single,hit_into_play,...,1.0,1.39,0.38,-0.38,30.5,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3189972,None,2026-05-29,NaN,NaN,NaN,"Vázquez, Christian",543877,682842,intent_walk,automatic_ball,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3189973,None,2026-05-29,NaN,NaN,NaN,"Vázquez, Christian",543877,682842,None,automatic_ball,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3189974,None,2026-05-29,NaN,NaN,NaN,"Cruz, Oneil",665833,607455,None,automatic_ball,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3189975,None,2026-05-29,NaN,NaN,NaN,"Realmuto, J.T.",592663,683618,None,automatic_ball,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_savant.shape)
print(df_savant["game_pk"].nunique())
print(df_savant["game_date"].min(), df_savant["game_date"].max())
print(df_savant["game_year"].value_counts().sort_index())

(3189977, 119)
8158
2023-03-30 2026-05-31
game_year
2023    956976
2024    944825
2025    948657
2026    339519
Name: count, dtype: int64


### <font color='00FFFF'> Listas de Variables Útiles

In [ ]:
leakage_cols = [
    "home_score", "away_score", "bat_score", "fld_score",
    "post_home_score", "post_away_score",
    "post_bat_score", "post_fld_score",
    "home_score_diff", "bat_score_diff",
    "home_win_exp", "bat_win_exp",
    "delta_home_win_exp", "delta_run_exp",
    "delta_pitcher_run_exp"
]

id_cols = [
    "game_pk", "game_date", "game_year",
    "home_team", "away_team",
    "inning_topbot"
]

candidate_cols = [
    "launch_speed",
    "launch_angle",
    "hit_distance_sc",
    "estimated_ba_using_speedangle",
    "estimated_woba_using_speedangle",
    "estimated_slg_using_speedangle",
    "woba_value",
    "babip_value",
    "iso_value",
    "release_speed",
    "release_spin_rate",
    "release_extension",
    "spin_axis",
    "arm_angle",
    "bat_speed",
    "swing_length",
    "n_thruorder_pitcher",
    "pitcher_days_since_prev_game",
    "batter_days_since_prev_game"
]



In [ ]:
cols_to_keep = id_cols + candidate_cols

df_savant = df_savant[cols_to_keep].copy()

### <font color='00FFFF'> Crear equipo bateador

In [ ]:
df_savant["batting_team"] = df_savant.apply(
    lambda row: row["away_team"] if row["inning_topbot"] == "Top" else row["home_team"],
    axis=1)

### <font color='00FFFF'> Agregar a nivel equipo-partido

In [ ]:
team_game_savant = (
    df_savant
    .groupby(["game_pk", "game_date", "game_year", "batting_team"])
    .agg({
        "launch_speed": "mean",
        "launch_angle": "mean",
        "hit_distance_sc": "mean",
        "estimated_ba_using_speedangle": "mean",
        "estimated_woba_using_speedangle": "mean",
        "estimated_slg_using_speedangle": "mean",
        "woba_value": "mean",
        "babip_value": "mean",
        "iso_value": "mean",
        "release_speed": "mean",
        "release_spin_rate": "mean",
        "release_extension": "mean",
        "bat_speed": "mean",
        "swing_length": "mean"
    })
    .reset_index()
)

In [ ]:
team_game_savant = team_game_savant.rename(columns={
    "batting_team": "team",
    "launch_speed": "team_avg_launch_speed_game",
    "launch_angle": "team_avg_launch_angle_game",
    "hit_distance_sc": "team_avg_hit_distance_game",
    "estimated_ba_using_speedangle": "team_avg_xba_game",
    "estimated_woba_using_speedangle": "team_avg_xwoba_game",
    "estimated_slg_using_speedangle": "team_avg_xslg_game",
    "woba_value": "team_avg_woba_game",
    "babip_value": "team_avg_babip_game",
    "iso_value": "team_avg_iso_game",
    "release_speed": "team_avg_release_speed_faced_game",
    "release_spin_rate": "team_avg_spin_rate_faced_game",
    "release_extension": "team_avg_release_extension_faced_game",
    "bat_speed": "team_avg_bat_speed_game",
    "swing_length": "team_avg_swing_length_game"
})

# <font color='00FF00'> Métricas históricas previas al juego

### <font color='00FFFF'> Ordenar

In [ ]:
team_game_savant["game_date"] = pd.to_datetime(team_game_savant["game_date"])

team_game_savant = team_game_savant.sort_values(
    ["team", "game_date", "game_pk"]
).reset_index(drop=True)

### <font color='00FFFF'> Crear variables de últimos 15 juegos

In [ ]:
rolling_features = [
    "team_avg_launch_speed_game",
    "team_avg_launch_angle_game",
    "team_avg_hit_distance_game",
    "team_avg_xba_game",
    "team_avg_xwoba_game",
    "team_avg_xslg_game",
    "team_avg_woba_game",
    "team_avg_babip_game",
    "team_avg_iso_game",
    "team_avg_bat_speed_game",
    "team_avg_swing_length_game"
]

In [ ]:
for col in rolling_features:
    team_game_savant[f"{col}_last15"] = (
        team_game_savant
        .groupby("team")[col]
        .transform(lambda x: x.shift(1).rolling(window=15, min_periods=5).mean())
    )

### <font color='00FFFF'> Validar

In [ ]:
team_game_savant[
    [
        "team",
        "game_date",
        "team_avg_xwoba_game",
        "team_avg_xwoba_game_last15",
        "team_avg_launch_speed_game",
        "team_avg_launch_speed_game_last15"
    ]
].head(30)

,team,game_date,team_avg_xwoba_game,team_avg_xwoba_game_last15,team_avg_launch_speed_game,team_avg_launch_speed_game_last15
0,ATH,2023-03-30,0.221117,NaN,83.161905,NaN
1,ATH,2023-04-01,0.345020,NaN,81.009302,NaN
2,ATH,2023-04-02,0.250365,NaN,78.106122,NaN
3,ATH,2023-04-03,0.338675,NaN,82.269841,NaN
4,ATH,2023-04-04,0.260472,NaN,79.522222,NaN
5,ATH,2023-04-05,0.344090,0.283130,79.673333,80.813879
6,ATH,2023-04-07,0.381276,0.293290,85.035556,80.623788
7,ATH,2023-04-08,0.210733,0.305859,78.328571,81.254040
8,ATH,2023-04-09,0.164930,0.293968,78.213158,80.888357
9,ATH,2023-04-10,0.280383,0.279631,82.605000,80.591112


### <font color='00FFFF'> Guardar dataset de equipo

In [ ]:
#team_game_savant.to_parquet("df_savant.parquet", index=False)

In [ ]:
df_savant = team_game_savant.copy()